#### 02 -- Order Item Options
EDA: Schema, keys, nulls, duplicates, option pricing and discounts

In [1]:
# Imports and setup

from pathlib import Path 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

project_root = Path.cwd().parent
raw_dir = project_root / "data" / "raw"

In [2]:
# Initialize Spark Session

spark = (
    SparkSession.builder
    .appName("CLVSourceProfiling")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/23 11:52:44 WARN Utils: Your hostname, SamDep, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/23 11:52:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 11:52:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Load order_item_options source data

order_options = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .csv(str(raw_dir / "order_item_options.csv"))
)

order_options.printSchema()

order_options.show(10, truncate=False)

print(f"Total records: {order_options.count():,}")

root
 |-- ORDER_ID: string (nullable = true)
 |-- LINEITEM_ID: string (nullable = true)
 |-- OPTION_GROUP_NAME: string (nullable = true)
 |-- OPTION_NAME: string (nullable = true)
 |-- OPTION_PRICE: double (nullable = true)
 |-- OPTION_QUANTITY: integer (nullable = true)

+------------------------+------------------------+-----------------+---------------+------------+---------------+
|ORDER_ID                |LINEITEM_ID             |OPTION_GROUP_NAME|OPTION_NAME    |OPTION_PRICE|OPTION_QUANTITY|
+------------------------+------------------------+-----------------+---------------+------------+---------------+
|6278eb58dfafab1e053fe172|6278eb63dfafab1e053fe18c|Cheese           |American Cheese|0.0         |1              |
|6278eb58dfafab1e053fe172|6278eb63dfafab1e053fe18c|Veggies          |Lettuce        |0.5         |1              |
|6278eb58dfafab1e053fe172|6278eb63dfafab1e053fe18c|Veggies          |Tomatoes       |0.0         |1              |
|6278eb58dfafab1e053fe172|6278eb63dfa

Total records: 193,017


In [4]:
# Profiling null values and candidate keys

# column-level null counts
order_options.select(
    [
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in order_options.columns
    ]
).show(vertical=True)

-RECORD 0----------------
 ORDER_ID          | 0   
 LINEITEM_ID       | 0   
 OPTION_GROUP_NAME | 0   
 OPTION_NAME       | 0   
 OPTION_PRICE      | 0   
 OPTION_QUANTITY   | 0   



In [5]:
# Key and record counts

order_options.select(
    F.count("*").alias("total_records"),
    F.countDistinct("ORDER_ID").alias("distinct_orders"),
    F.countDistinct(
        "ORDER_ID", "LINEITEM_ID"
    ).alias("distinct_order_items")
).show()

+-------------+---------------+--------------------+
|total_records|distinct_orders|distinct_order_items|
+-------------+---------------+--------------------+
|       193017|          78614|              102712|
+-------------+---------------+--------------------+



In [6]:
# Duplicate records

order_options.groupBy(
    *order_options.columns
).count().filter(
    F.col("count") > 1
).show(truncate=False)

+------------------------+------------------------+-----------------+------------+------------+---------------+-----+
|ORDER_ID                |LINEITEM_ID             |OPTION_GROUP_NAME|OPTION_NAME |OPTION_PRICE|OPTION_QUANTITY|count|
+------------------------+------------------------+-----------------+------------+------------+---------------+-----+
|5f71f5fcadb0d5cc0a7b2499|5f71f5fc37ab4656377b245f|Brooklyn Bliss   |No          |0.0         |1              |3    |
|5f185365505ee93e7aa25f76|5f18536b37ab46ea287b23f8|Vegetables       |Cilantro    |0.0         |1              |2    |
|5f6a531badb0d5b62b82f89e|5f6a5321505ee9553034f194|Vegetables       |Escarole    |0.0         |1              |2    |
|5f6b95b5505ee94655340ba1|5f6b95ba37ab463a568b6351|Shakshuka        |No          |0.0         |1              |3    |
|5f71b48b505ee9e10fa8bc2d|5f71b4a0505ee9170ea8bc4b|Milk Options     |No          |0.0         |1              |5    |
|5f1fd344505ee92b13d0d281|5f1fd344505ee92b13d0d282|Veget

In [7]:
# Establish relationship grain

options_per_item = (
    order_options
    .groupBy("ORDER_ID", "LINEITEM_ID")
    .agg(
        F.count("*").alias("option_count")
    )
)

options_per_item.agg(
    F.min("option_count").alias("min_options"),
    F.max("option_count").alias("max_options"),
    F.avg("option_count").alias("avg_options")
).show()

+-----------+-----------+------------------+
|min_options|max_options|       avg_options|
+-----------+-----------+------------------+
|          1|        152|1.8792059350416699|
+-----------+-----------+------------------+



In [8]:
# Examine pricing and quantities

order_options.select(
    F.min("OPTION_PRICE").alias("min_price"),
    F.max("OPTION_PRICE").alias("max_price"),
    F.sum(
        (F.col("OPTION_PRICE") < 0).cast("int")
    ).alias("negative_prices"),
    F.sum(
        (F.col("OPTION_PRICE") == 0).cast("int")
    ).alias("zero_prices"),
    F.min("OPTION_QUANTITY").alias("min_quantity"),
    F.max("OPTION_QUANTITY").alias("max_quantity")
).show()

+---------+---------+---------------+-----------+------------+------------+
|min_price|max_price|negative_prices|zero_prices|min_quantity|max_quantity|
+---------+---------+---------------+-----------+------------+------------+
|      0.0|      8.0|              0|     127980|           1|           1|
+---------+---------+---------------+-----------+------------+------------+



In [9]:
# Quantify exact duplicate option groups and records beyond the first occurrence

duplicate_summary = (
    order_options
    .groupBy(*order_options.columns)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_summary.select(
    F.count("*").alias("duplicate_groups"),
    F.sum(F.col("count") - 1).alias("additional_records")
).show()

+----------------+------------------+
|duplicate_groups|additional_records|
+----------------+------------------+
|             616|              2299|
+----------------+------------------+



In [10]:
# Identify order items with unusually high numbers of option records

order_options.groupBy(
    "ORDER_ID", "LINEITEM_ID"
).count().orderBy(
    F.desc("count")
).show(10, truncate=False)

+------------------------+------------------------+-----+
|ORDER_ID                |LINEITEM_ID             |count|
+------------------------+------------------------+-----+
|5ed7ab56505ee9030f7b23d0|5ed7ab59505ee96c0f7b23c8|152  |
|5ed7b3b6902ad5c4357b23ee|5ed7b479902ad5dd377b23e2|148  |
|5ed7aa37902ad58b237b23ef|5ed7ab55902ad5e7287b23cf|148  |
|5f8571404f5ee97f0b211fe3|5f857286505ee9bd0607e190|144  |
|5f859d1e505ee9d93f07e1b6|5f85ae374f5ee95c647b23d7|144  |
|5ed7b45c902ad514387b23d3|5ed7b45f902ad54c3a7b23c8|143  |
|5f843e1e4f5ee9c1577b23cd|5f843e20505ee9f13c7b23e2|140  |
|5f844149505ee9eb437b23c9|5f844149505ee9eb437b23ca|140  |
|5f8951ba505ee990627b23f9|5f8954664f5ee9fe407b23c8|103  |
|5ece77ff902ad501337b23ff|5ece7801902ad537327b2415|73   |
+------------------------+------------------------+-----+
only showing top 10 rows


In [11]:
# Profile positively priced options and their frequency

order_options.filter(
    F.col("OPTION_PRICE") > 0
).groupBy(
    "OPTION_GROUP_NAME",
    "OPTION_NAME",
    "OPTION_PRICE"
).count().orderBy(
    F.desc("count")
).show(20, truncate=False)

+----------------------------+-----------------------------+------------+-----+
|OPTION_GROUP_NAME           |OPTION_NAME                  |OPTION_PRICE|count|
+----------------------------+-----------------------------+------------+-----+
|Smoothie Add-ons            |Add Protein                  |1.0         |2703 |
|Bacon Egg And Cheese Options|No Brioche, Substitute Bagel |1.0         |2493 |
|Bacon Egg And Cheese Options|Add Avocado                  |1.0         |2165 |
|Breakfast Burrito Options   |Add Breakfast Chicken Sausage|1.0         |1999 |
|Bacon Egg And Cheese Options|Add Breakfast Chicken Sausage|1.0         |1927 |
|Bacon Egg And Cheese Options|Add Gluten Free Bread        |1.0         |1820 |
|Breakfast Burrito Options   |Add Smokehouse Bacon         |2.0         |1562 |
|Breakfast Burrito Options   |Add Egg                      |1.0         |1270 |
|Chop Salad Options          |Add Grilled Chicken          |2.0         |1259 |
|Cracked Oats Oatmeal Options|Add Choppe

In [13]:
# Parent items

order_items = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .csv(str(raw_dir / "order_items.csv"))
)

In [14]:
# Compare option-heavy line items with their parent item quantities and prices

option_counts = (
    order_options
    .groupBy("ORDER_ID", "LINEITEM_ID")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

option_counts.join(
    order_items.select(
        "ORDER_ID",
        "LINEITEM_ID",
        "ITEM_NAME",
        "ITEM_PRICE",
        "ITEM_QUANTITY"
    ),
    on=["ORDER_ID", "LINEITEM_ID"],
    how="left"
).show(truncate=False)

+------------------------+------------------------+-----+----------------+----------+-------------+
|ORDER_ID                |LINEITEM_ID             |count|ITEM_NAME       |ITEM_PRICE|ITEM_QUANTITY|
+------------------------+------------------------+-----+----------------+----------+-------------+
|5f8951ba505ee990627b23f9|5f8954664f5ee9fe407b23c8|103  |Killer Cobb | GF|22.5      |1            |
|5ed7ab56505ee9030f7b23d0|5ed7ab59505ee96c0f7b23c8|152  |Middle Eastern  |51.5      |1            |
|5ed7aa37902ad58b237b23ef|5ed7ab55902ad5e7287b23cf|148  |Middle Eastern  |50.5      |1            |
|5f8571404f5ee97f0b211fe3|5f857286505ee9bd0607e190|144  |Killer Cobb | GF|190.0     |10           |
|5ece77ff902ad501337b23ff|5ece7801902ad537327b2415|73   |Buddha Bowl     |22.0      |1            |
|5f859d1e505ee9d93f07e1b6|5f85ae374f5ee95c647b23d7|144  |Killer Cobb | GF|18.0      |1            |
|5f843e1e4f5ee9c1577b23cd|5f843e20505ee9f13c7b23e2|140  |Killer Cobb | GF|127.0     |10           |
